# OdyssNet-LLM Cloud Trainer
- This IPYNB file written to train OdyssNet based LLM's on Google Collab or Google Cloud, paths kept Collab friendly and root relative, AI checkpoints loaded/saved from Google Drive. Settings(neuron_count, sequence length etc.) kept high for infinite cloud training.

In [ ]:
!git clone https://github.com/theomgdev/OdyssNet.git
!pip install -r OdyssNet/requirements.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
import sys
import os
import time
import random
from typing import Any, Mapping, cast
from tokenizers import Tokenizer
from tokenizers.implementations import ByteLevelBPETokenizer

try:
    import tiktoken
except ImportError:
    pass

# --- ENVIRONMENT & IMPORTS ---
PROJECT_ROOT_CANDIDATES = [
    "/content/OdyssNet",
    os.getcwd(),
]
for _cand in PROJECT_ROOT_CANDIDATES:
    if os.path.isdir(_cand) and _cand not in sys.path:
        sys.path.append(os.path.abspath(_cand))

from odyssnet import (
    OdyssNet,
    OdyssNetTrainer,
    save_checkpoint,
    load_checkpoint,
    transplant_weights,
    set_seed,
)
from datasets import load_dataset

# TF32 Optimization
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# --- CONFIGURATION (Cloud-Scaled) ---
TRUNCATED_BPTT_SEQ_LEN = 16
GENERATION_LENGTH = 128
SEQ_LEN = 512
BATCH_SIZE = -1
STEPS_PER_EPOCH = 100
LOG_INTERVAL = 1
MAX_START_SKIP = 1000
RESET_DATA_ITER = False

NUM_NEURONS = 8192
INPUT_NEURON_COUNT = 128
OUTPUT_NEURON_COUNT = 128
ACTIVATION = ['none', 'gelu_tanh', 'tanh', 'none']
WEIGHT_INIT = ['quiet', 'resonant', 'quiet', 'zero']
GATES = ['none', 'none', 'identity']
THINK_GAP = 5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# NEUROGENESIS CONFIG
NEUROGENESIS_ENABLED = False
MAX_LOSS_INCREASE = 10
NEUROGENESIS_AMOUNT = 10

# REGENERATION CONFIG (PHOENIX)
DARWINIAN_REGENERATION = False
REGENERATION_MODE = 'percentage'
REGENERATION_THRESHOLD = 0.01
REGENERATION_PERCENTAGE = 0.001
REGENERATION_INTERVAL = 10

# TOKENIZER CONFIG
USE_TIKTOKEN = False
TIKTOKEN_ENCODING = "o200k_base"
CUSTOM_VOCAB_SIZE = 1024

# OPTIMIZER CONFIG
RESET_OPTIMIZER_ON_LOAD = False
OVERWRITE_LR_OF_CKPT = True
LEARNING_RATE = 3e-4

# TIE EMBEDDINGS (VRAM Saving & Parameter Sharing)
TIE_EMBEDDINGS = False

# --- COLAB DRIVE SETUP ---
COLAB_DRIVE_PATH = '/content/drive/MyDrive/OdyssNet'
CKPT_DIR = os.path.join(COLAB_DRIVE_PATH, 'ckpt')
os.makedirs(CKPT_DIR, exist_ok=True)


def get_or_train_tokenizer():
    k_size = CUSTOM_VOCAB_SIZE // 1000
    tokenizer_path = os.path.join(CKPT_DIR, f"poc_tokenizer_{k_size}k.json")

    if os.path.exists(tokenizer_path):
        print(f"📚 Loading existing {k_size}k BPE Tokenizer from {tokenizer_path}...")
        return Tokenizer.from_file(tokenizer_path)

    print(f"📚 Training new {k_size}k BPE Tokenizer from data slice...")
    tokenizer = ByteLevelBPETokenizer()
    dataset = load_dataset("roneneldan/TinyStories", split="train", streaming=True)

    texts: list[str] = []
    count = 0
    for item in dataset:
        row = cast(Mapping[str, Any], item)
        texts.append(str(row.get('text', '')))
        count += 1
        if count >= 100000:
            break

    tokenizer.train_from_iterator(
        iter(texts),
        vocab_size=CUSTOM_VOCAB_SIZE,
        min_frequency=2,
        special_tokens=["<s>", "<pad>", "</s>", "<unk>", "<mask>"],
    )

    tokenizer.save(tokenizer_path)
    print(f"✅ Tokenizer saved to {tokenizer_path}")
    return tokenizer

class TiktokenWrapper:
    def __init__(self, encoding_name="o200k_base"):
        try:
            import tiktoken as _tiktoken
        except ImportError:
            raise ImportError("tiktoken is not installed. Please `pip install tiktoken` or set USE_TIKTOKEN = False.")
        self.tokenizer = _tiktoken.get_encoding(encoding_name)
        self.eos_id = self.tokenizer.eot_token

    def encode(self, text):
        class Encoded:
            def __init__(self, ids):
                self.ids = ids
        return Encoded(self.tokenizer.encode(text, allowed_special="all"))

    def decode(self, ids):
        return self.tokenizer.decode(ids)

    def get_vocab_size(self):
        return self.tokenizer.n_vocab

    def token_to_id(self, token):
        if token in ("</s>", "<|endoftext|>"):
            return self.eos_id
        try:
            return self.tokenizer.encode(token, allowed_special="all")[0]
        except Exception:
            return self.eos_id

def get_tokenizer():
    if USE_TIKTOKEN:
        print(f"📚 Loading Tiktoken tokenizer: {TIKTOKEN_ENCODING}...")
        return TiktokenWrapper(TIKTOKEN_ENCODING)
    return get_or_train_tokenizer()

TOKENIZER = get_tokenizer()
VOCAB_SIZE = TOKENIZER.get_vocab_size()

class TinyStoriesIterableDataset(torch.utils.data.IterableDataset):
    def __init__(self, seq_len, tokenizer, skip_offset=0, debug=False):
        self.seq_len = seq_len
        self.tokenizer = tokenizer
        self.skip_offset = skip_offset
        self.debug = debug
        self.current_doc_index = skip_offset
        print("🌊 Connecting to the dataset...")
        self.dataset: Any = load_dataset("roneneldan/TinyStories", split="train", streaming=True)

    def __iter__(self):
        start_skip = self.skip_offset

        if start_skip == 0 or RESET_DATA_ITER:
            start_skip = random.randint(0, MAX_START_SKIP)
            print(f"🔀 Random Start: Skipping {start_skip} documents...")
        else:
            print(f"⏩ Resuming from Document #{start_skip}...")

        local_doc_index = start_skip

        if start_skip > 0:
            iterator = iter(cast(Any, self.dataset).skip(start_skip))
        else:
            iterator = iter(cast(Any, self.dataset))

        buffer_tokens = []

        while True:
            while len(buffer_tokens) < self.seq_len + 1:
                try:
                    item = next(iterator)
                    local_doc_index += 1
                    if self.debug and local_doc_index % 1000 == 0:
                        print(f"📊 Streaming Index: Document #{local_doc_index}")
                    row = cast(Mapping[str, Any], item)
                    text = str(row.get('text', ''))
                    encoded = self.tokenizer.encode(text)
                    buffer_tokens.extend(encoded.ids)
                    buffer_tokens.append(self.tokenizer.token_to_id("</s>"))
                except StopIteration:
                    iterator = iter(self.dataset)
                    local_doc_index = 0

            chunk_tokens = buffer_tokens[:self.seq_len + 1]
            buffer_tokens = buffer_tokens[self.seq_len + 1:]

            if len(chunk_tokens) == self.seq_len + 1:
                x = torch.tensor(chunk_tokens[:-1], dtype=torch.long)
                y = torch.tensor(chunk_tokens[1:], dtype=torch.long)
                yield x, y, local_doc_index

    def get_vocab_size(self):
        return self.tokenizer.get_vocab_size()

def generate(model, tokenizer, start_str="The", length=None, temperature=0.8, top_k=40, top_p=0.9):
    if length is None:
        length = GENERATION_LENGTH

    model.eval()
    encoded = tokenizer.encode(start_str)
    input_seq = encoded.ids
    current_state = None

    x_in = torch.tensor(input_seq, dtype=torch.long, device=model.device).unsqueeze(0)
    steps_total = x_in.shape[1] * (THINK_GAP + 1)

    with torch.no_grad():
        _, current_state = model(x_in, steps=steps_total)
        last_token_idx = input_seq[-1]

        for _ in range(length):
            x_next = torch.tensor([[last_token_idx]], dtype=torch.long, device=model.device)
            preds, current_state = model(x_next, steps=1 + THINK_GAP, current_state=current_state)
            logits = preds[0, 0, :]

            if temperature > 0:
                logits = logits / temperature

            if top_k is not None and top_k > 0:
                v, _ = torch.topk(logits, min(top_k, len(logits)))
                logits[logits < v[-1]] = float('-inf')

            if top_p is not None and 0 < top_p < 1.0:
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
                sorted_indices_to_remove = cumulative_probs > top_p
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0
                indices_to_remove = sorted_indices_to_remove.scatter(0, sorted_indices, sorted_indices_to_remove)
                logits[indices_to_remove] = float('-inf')

            probs = torch.softmax(logits, dim=0)
            if torch.isnan(probs).any() or torch.sum(probs) == 0:
                probs = torch.ones_like(probs) / len(probs)

            next_idx = torch.multinomial(probs, 1).item()
            input_seq.append(next_idx)
            last_token_idx = next_idx

    return tokenizer.decode(input_seq)


def initialize_system(vocab_size, num_neurons, device, input_count=-1, output_count=-1, lr=1e-4, activation=None, weight_init=None, gates=None, hook=None):
    if input_count == -1:
        input_neuron_count = num_neurons // 2
    else:
        input_neuron_count = input_count

    if output_count == -1:
        output_neuron_count = num_neurons // 2
    else:
        output_neuron_count = output_count

    input_ids = list(range(input_neuron_count))
    output_ids = list(range(input_neuron_count, input_neuron_count + output_neuron_count))

    model = OdyssNet(
        num_neurons=num_neurons,
        input_ids=input_ids,
        output_ids=output_ids,
        device=device,
        activation=activation,
        weight_init=weight_init,
        gate=gates,
        gradient_checkpointing=True,
        vocab_size=vocab_size,
        vocab_mode='discrete',
        tie_embeddings=TIE_EMBEDDINGS
    )

    trainer = OdyssNetTrainer(
        model,
        lr=lr,
        device=device,
        gradient_persistence=0.0,
        anomaly_hook=hook,
    )

    return model, trainer, input_ids, output_ids


def calculate_optimal_batch_size(model, device, seq_len, think_gap, truncated_bptt_seq_len):
    print("\n⚖️  Auto-Tuning Batch Size...")

    if device == 'cpu':
        return 32

    if device == 'cuda':
        t = torch.cuda.get_device_properties(0).total_memory
        a = torch.cuda.memory_allocated(0)

        actual_total_params = model.get_num_params()
        baseline_vram = (actual_total_params * 16) + (1.5 * 1024 * 1024 * 1024)

        num_neurons = model.num_neurons
        if hasattr(model, 'vocab_size') and model.vocab_size is not None:
            vocab_dim = model.vocab_size[-1] if isinstance(model.vocab_size, (list, tuple)) else model.vocab_size
        else:
            vocab_dim = num_neurons

        if truncated_bptt_seq_len > 0:
            effective_mem_len = truncated_bptt_seq_len * (think_gap + 1)
            logit_seq_len = truncated_bptt_seq_len
        else:
            effective_mem_len = seq_len * (think_gap + 1)
            logit_seq_len = seq_len

        core_mem_per_sample = effective_mem_len * num_neurons * 8
        logit_mem_per_sample = logit_seq_len * vocab_dim * 12
        vram_per_sample = (core_mem_per_sample + logit_mem_per_sample) * 1.1

        free_vram = t - a - baseline_vram
        safe_vram = free_vram * 0.9

        calc_batch = int(safe_vram / vram_per_sample) if vram_per_sample > 0 else 1
        calc_batch = max(1, calc_batch)
        if calc_batch > 8:
            calc_batch = (calc_batch // 8) * 8

        print(f"   VRAM Total:  {t / 1e9:.2f} GB")
        print(f"   Model Base:  {baseline_vram / 1e9:.2f} GB (Params + Optim)")
        print(f"   VRAM Free:   {free_vram / 1e9:.2f} GB")
        print(f"   Mem/Sample:  {vram_per_sample / 1e6:.2f} MB")
        print(f"   Opt. Batch:  {calc_batch}")

        return calc_batch

    return 32

def main():
    global NUM_NEURONS, BATCH_SIZE
    set_seed(42)

    print("🚀 OdyssNet-LLM (TinyStories Streaming) - NATIVE THINKING MODE")
    print("--- Configuration ---")
    print(f"SEQ_LEN: {SEQ_LEN}")
    print(f"BATCH_SIZE: {BATCH_SIZE} (Will Auto-Tune if -1)")
    print(f"NUM_NEURONS: {NUM_NEURONS}")
    print(f"INPUT_NEURON_COUNT: {INPUT_NEURON_COUNT} (Half of NumNeurons if -1)")
    print(f"OUTPUT_NEURON_COUNT: {OUTPUT_NEURON_COUNT} (Half of NumNeurons if -1)")
    print(f"TRUNCATED_BPTT_SEQ_LEN (Tokens): {TRUNCATED_BPTT_SEQ_LEN}")
    print(f"STEPS_PER_EPOCH: {STEPS_PER_EPOCH}")
    print(f"LOG_INTERVAL: {LOG_INTERVAL}")
    print(f"MAX_START_SKIP: {MAX_START_SKIP}")
    print(f"RESET_DATA_ITER: {RESET_DATA_ITER}")
    print(f"GENERATION_LENGTH: {GENERATION_LENGTH}")
    print(f"THINK_GAP: {THINK_GAP}")
    print(f"ACTIVATION: {ACTIVATION}")
    print(f"WEIGHT_INIT: {WEIGHT_INIT}")
    print(f"GATES: {GATES}")
    print(f"TIE_EMBEDDINGS: {TIE_EMBEDDINGS}")
    tiktoken_info = f", Encoding: {TIKTOKEN_ENCODING}" if USE_TIKTOKEN else ""
    print(f"VOCAB_SIZE: {VOCAB_SIZE} (Tiktoken: {USE_TIKTOKEN}{tiktoken_info})")
    print(f"DEVICE: {DEVICE}")
    print(f"NEUROGENESIS: Enabled={NEUROGENESIS_ENABLED}, MaxLossInc={MAX_LOSS_INCREASE}, Amount={NEUROGENESIS_AMOUNT}")
    if DARWINIAN_REGENERATION:
        regen_val = f"{REGENERATION_PERCENTAGE:.2%}" if REGENERATION_MODE == 'percentage' else f"{REGENERATION_THRESHOLD}"
        print(f"PHOENIX (Regeneration): Mode={REGENERATION_MODE}, Val={regen_val}, Interval={REGENERATION_INTERVAL}")
    else:
        print("PHOENIX (Regeneration): Disabled")
    print(f"RESET_OPTIM_ON_LOAD: {RESET_OPTIMIZER_ON_LOAD}")
    print(f"LEARNING_RATE: {LEARNING_RATE}")
    print(f"OVERWRITE_LR_OF_CKPT: {OVERWRITE_LR_OF_CKPT}")
    print("---------------------")

    core_act_name = ACTIVATION[1] if isinstance(ACTIVATION, list) else ACTIVATION
    ckpt_path = os.path.join(CKPT_DIR, f'llm_tinystories_{core_act_name}_latest.pth')
    ckpt_best_path = os.path.join(CKPT_DIR, f'llm_tinystories_{core_act_name}_best.pth')

    resume_doc_index = 0
    start_epoch = 0

    if os.path.exists(ckpt_path):
        try:
            peek = torch.load(ckpt_path, map_location='cpu')
            resume_doc_index = peek.get('dataset_step', 0)
            if resume_doc_index > 0:
                print(f"📂 Resuming dataset from index: {resume_doc_index}")
            start_epoch = peek.get('epoch', -1) + 1
        except Exception:
            pass

    dataset = TinyStoriesIterableDataset(SEQ_LEN, TOKENIZER, skip_offset=resume_doc_index, debug=False)
    hook_state = {'increase_count': 0}

    def anomaly_event_handler(anomaly_type, loss_val):
        if anomaly_type == "increase" and NEUROGENESIS_ENABLED:
            hook_state['increase_count'] += 1
            if hook_state['increase_count'] >= MAX_LOSS_INCREASE:
                print(f"\n🧬 [ANOMALY HOOK] Loss increase limit reached ({MAX_LOSS_INCREASE}). Expanding Network (Neurogenesis)...")
                trainer.expand(amount=NEUROGENESIS_AMOUNT)
                global NUM_NEURONS
                NUM_NEURONS = model.num_neurons
                hook_state['increase_count'] = 0

    model, trainer, input_ids, output_ids = initialize_system(
        VOCAB_SIZE,
        NUM_NEURONS,
        DEVICE,
        input_count=INPUT_NEURON_COUNT,
        output_count=OUTPUT_NEURON_COUNT,
        lr=LEARNING_RATE,
        activation=ACTIVATION,
        weight_init=WEIGHT_INIT,
        gates=GATES,
        hook=anomaly_event_handler,
    )
    NUM_NEURONS = model.num_neurons

    print(f"Input IDs: {input_ids[0]}-{input_ids[-1]}")
    print(f"Output IDs: {output_ids[0]}-{output_ids[-1]}")

    def restore_checkpoint_runtime():
        opt_arg = None if RESET_OPTIMIZER_ON_LOAD else trainer.optimizer
        target_lr = LEARNING_RATE if OVERWRITE_LR_OF_CKPT else None
        checkpoint_data = load_checkpoint(model, opt_arg, ckpt_path, device=DEVICE, strict=True, lr=target_lr)

        trainer_state = checkpoint_data.get('trainer_state_dict')
        if trainer_state is not None:
            try:
                trainer.load_state_dict(trainer_state)
                print("🧠 Trainer runtime state restored (scaler/accumulators).")
            except Exception as e:
                print(f"⚠️ Could not restore trainer runtime state: {e}")

        return checkpoint_data

    if os.path.exists(ckpt_path):
        try:
            ckpt_peek = torch.load(ckpt_path, map_location=DEVICE)
            if 'model_state_dict' in ckpt_peek and 'W' in ckpt_peek['model_state_dict']:
                saved_dim = ckpt_peek['model_state_dict']['W'].shape[0]
                if saved_dim != NUM_NEURONS:
                    print("\n⚠️ ARCHITECTURE MISMATCH DETECTED!")
                    print(f"   Current Model: {NUM_NEURONS}")
                    print(f"   Checkpoint:    {saved_dim}")
                    print("Attempting safe transplant to current cloud config...")
                    transplant_weights(model, ckpt_path, device=DEVICE)
                    print("🧬 Transplant complete.")
                else:
                    print(f"🔄 Loading Checkpoint from {ckpt_path}...")
                    restore_checkpoint_runtime()
                    print(f"✅ Resuming from Epoch {start_epoch}")
            else:
                loaded_ckpt = restore_checkpoint_runtime()
                start_epoch = loaded_ckpt.get('epoch', 0) + 1
        except Exception as e:
            print(f"⚠️ Failed to load/inspect checkpoint: {e}. Starting fresh.")

    if BATCH_SIZE == -1:
        BATCH_SIZE = calculate_optimal_batch_size(
            model,
            DEVICE,
            SEQ_LEN,
            THINK_GAP,
            TRUNCATED_BPTT_SEQ_LEN,
        )

    num_workers = 1
    dataloader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        num_workers=num_workers,
        prefetch_factor=4,
        persistent_workers=(num_workers > 0),
        pin_memory=True,
    )

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    trainer.loss_fn = criterion

    def flatten_logits(out):
        return out.reshape(-1, dataset.get_vocab_size())

    print("\n--- MODEL INFO ---")
    print(f"Total Trainable Parameters: {model.get_num_params():,}")

    print("\n--- GENERATION PREVIEW ---")
    try:
        gen_text = generate(model, TOKENIZER, start_str="Once upon a time")
        print(f"Sample: {gen_text}\n")
    except Exception as e:
        print(f"Error: {e}")

    print("--- TRAINING LOOP ---")

    epoch = start_epoch
    best_loss = float('inf')

    if os.path.exists(ckpt_best_path):
        try:
            best_ckpt = torch.load(ckpt_best_path, map_location=DEVICE)
            best_loss = best_ckpt.get('loss', float('inf'))
            print(f"🏆 Historical Best Loss: {best_loss:.4f}")
        except Exception:
            pass

    data_iterator = iter(dataloader)

    while True:
        total_loss = 0
        steps = 0
        start_time = time.time()

        for batch_idx in range(STEPS_PER_EPOCH):
            try:
                x, y, current_doc_tensor = next(data_iterator)
                current_doc = current_doc_tensor[-1].item()
                dataset.current_doc_index = current_doc
            except StopIteration:
                print("🔄 Restarting iterator...")
                data_iterator = iter(dataloader)
                x, y, current_doc_tensor = next(data_iterator)
                current_doc = current_doc_tensor[-1].item()
                dataset.current_doc_index = current_doc

            x = x.to(DEVICE)
            y = y.to(DEVICE)
            y_flat = y.reshape(-1)

            seq_len = x.shape[1]
            total_thinking_steps = seq_len * (THINK_GAP + 1)

            if TRUNCATED_BPTT_SEQ_LEN != -1 and TRUNCATED_BPTT_SEQ_LEN > 0:
                current_state = None
                batch_loss = 0
                steps_count = 0
                chunk_len = TRUNCATED_BPTT_SEQ_LEN

                for t_start in range(0, seq_len, chunk_len):
                    t_end = min(t_start + chunk_len, seq_len)
                    x_chunk = x[:, t_start:t_end]
                    y_chunk_flat = y[:, t_start:t_end].reshape(-1)
                    actual_tokens = t_end - t_start
                    chunk_thinking_steps = actual_tokens * (THINK_GAP + 1)

                    loss, current_state = trainer.train_batch(
                        x_chunk,
                        y_chunk_flat,
                        thinking_steps=chunk_thinking_steps,
                        full_sequence=True,
                        output_transform=flatten_logits,
                        initial_state=current_state,
                        return_state=True
                    )

                    current_state = current_state.detach()
                    batch_loss += loss
                    steps_count += 1

                loss = batch_loss / max(steps_count, 1)
            else:
                loss = trainer.train_batch(
                    x,
                    y_flat,
                    thinking_steps=total_thinking_steps,
                    full_sequence=True,
                    output_transform=flatten_logits
                )

            current_lr = trainer.optimizer.param_groups[0]['lr']

            total_loss += loss
            steps += 1

            if batch_idx % LOG_INTERVAL == 0:
                loss_val = loss.item() if isinstance(loss, torch.Tensor) else loss
                ppl = np.exp(loss_val)
                print(f"Epoch {epoch} | Batch {batch_idx} | Doc #{current_doc} | Loss {loss:.4f} | PPL {ppl:.2f} | LR {current_lr:.2e}")

        avg_loss = total_loss / steps
        avg_loss_val = avg_loss.item() if isinstance(avg_loss, torch.Tensor) else avg_loss
        avg_ppl = np.exp(avg_loss_val)
        print(f"Epoch {epoch} Completed | Avg Loss: {avg_loss:.4f} | Avg PPL {avg_ppl:.2f} | Time: {time.time() - start_time:.1f}s")

        print("--- GENERATION ---")
        try:
            gen_text = generate(model, TOKENIZER, start_str="Once upon a time ")
            print(gen_text)
        except Exception as e:
            print(f"Generation Error: {e}")
        print("------------------")

        ckpt_extra_data = {
            'initial_lr': trainer.initial_lr,
            'dataset_step': dataset.current_doc_index,
            'trainer_state_dict': trainer.state_dict(),
        }

        save_checkpoint(model, trainer.optimizer, epoch, avg_loss, ckpt_path, extra_data=ckpt_extra_data)
        print(f"💾 Checkpoint Saved: {ckpt_path} (Doc Index: {dataset.current_doc_index})")

        if avg_loss < best_loss:
            best_loss = avg_loss
            save_checkpoint(model, trainer.optimizer, epoch, avg_loss, ckpt_best_path, extra_data=ckpt_extra_data)
            print(f"🏆 NEW RECORD! Saved: {ckpt_best_path} (Loss: {best_loss:.4f})")

        if DARWINIAN_REGENERATION and epoch % REGENERATION_INTERVAL == 0:
            print("🔥 Phoenix Protocol: Checking for dead synapses...")
            p_arg = REGENERATION_PERCENTAGE if REGENERATION_MODE == 'percentage' else None
            t_arg = REGENERATION_THRESHOLD
            revived, total = trainer.regenerate_synapses(threshold=t_arg, percentage=p_arg)
            if revived > 0:
                print(f"🔥 Reborn: {revived}/{total} ({revived/total:.2%}) synapses regenerated.")

        epoch += 1

if __name__ == "__main__":
    main()